# Scikit-Fuzzy — Desarrollo (Grupo CCCV)
### IPD434 — Computación Evolutiva e Inteligencia Artificial
**Universidad Técnica Federico Santa María**

---

## 🎯 Objetivos de Aprendizaje

Al finalizar esta clase, el estudiante será capaz de:

1. **Aplicar** `scikit-fuzzy` para modelar el problema de control difuso propuesto.
2. **Definir** las variables de entrada/salida, sus funciones de membresía y el conjunto de reglas.
3. **Simular** el sistema y defuzzificar la salida para casos concretos.
4. **Interpretar** la superficie de control obtenida y evaluar el diseño.

> 🧪 **Cuaderno de desarrollo (grupo CCCV):** resolución guiada del ejercicio del módulo.

> 💡 **Prerrequisitos:** [02_SciKit_Fuzzy](02_SciKit_Fuzzy.ipynb).

## Ejercicio 1: Aire acondicionado — solución (CCCV)

A continuación se desarrolla, paso a paso, un controlador difuso que en modo **auto** decide el **modo** de operación y la **velocidad** del ventilador. Primero se resuelve la parte 1 (diseño del FIS) y luego la parte 2 (simulación temporal).

Ud. ha sido contratado por IPD-Electrónics para trabajar en su marca TEL-Air que comercializará equipos de aire acondicionado. Estos dispositivos climatizan ambientes interiores en el rango entre $17ºC$ a $30ºC$, lanzando aire frio o aire caliente dependiendo de las instrucciones que el usuario le entregue, hasta llegar a la temperatura deseada. Así:

* Para climatizar utilizando el modo _calor_, el aire acondicionado lanza aire caliente constante, a unos $42ºC$, hasta llegar a la temperatura deseada. 
* Para climatizar utilizando el modo _frío_, el aire acondicionado lanza aire fresco constante, a unos $13ºC$, hasta llegar a la temperatura deseada.
* Para mantener el clima actual del ambiente se utiliza el modo _stand by_, que provee de circulación del aire sin modificar la variación normal de la temperatura.
* Para climatizar automáticamente se utiliza el modo _auto_, el cual se encarga de escoger el modo del aire acondicionado, y el nivel de velocidad utilizado.

El aire es lanzado por los dispositivos en tres posibles velocidades: baja, media o alta. Estas velocidades modifican la temperatura, según el modo de aire y las características de la zona climatizada, en los rangos (en $\frac{ºC}{\text{min}}$) $[0.1, 0.7]$, $[0.5, 1.2]$ y $[0.8, 1.9]$, respectivamente. Considere que si está en modo _stand by_ estas variaciones se darán para acercarse a la temperatura del exterior.  

Para controlar la temperatura del ambiente, los dispositivos cuentan con un termostato que les indica la temperatura actual de la zona climatizada.

Considere que para limitar el consumo energético, el modo _auto_ define las siguientes reglas bases:
* Si la diferencia de temperatura es negativa, debe utilizar modo calor.
* Si la diferencia de temperatura es positiva, debe utilizar modo frío.
* Si no hay diferencia de temperatura, debe utilizar el modo stand by.
* Si se encuentra cercano o en la temperatura deseada, utilizar el modo velocidad baja.
* Si se encuentra muy alejado en la temperatura, debe utilizar el modo de velocidad alta. 

Y que además, utilizando el historico de temperatura actual y de destino, reflejado en las configuraciones manuales, se ha determinado qué:
* La temperatura puede ser catalogada como fría si está bajo el umbral de los 15ºC.
* La temperatura puede ser catalogada como fresca si está en el rango de los 12.5ºC a los 20ºC.
* La temperatura puede ser catalogada como ambiente entre los 18ºC a los 25ºC.
* La temperatura puede ser catalogada como calurosa sobre el rango de los 23ºC.

Y como es un sistema basado en la percepción y el comfort humano, se definen las siguientes reglas básicas:
* Si la temperatura es fría, el aire debe utilizar modo calor. 
* Si la temperatura es calurosa, el aire debe utilizar modo frío.

1. Genere un sistema de inferencia difusa que permita para un instante determinado, utilizando el modo _auto_, determinar el modo de función del dispositivo y la velocidad con la cual lanza el aire. Determine claramente los antecendentes y los consecuentes del sistema, funciones de membresía y las reglas de inferencia difusa del sistema.

### Enfoque de la solución (CCCV)

Modelamos el controlador con **dos antecedentes** y **dos consecuentes**:

| Rol | Variable | Universo | Etiquetas |
|-----|----------|----------|-----------|
| Antecedente | `dif` = T actual − T deseada | $[-10, 10]$ | negativa, cercana, positiva |
| Antecedente | `temp` (temperatura actual) | $[0, 40]$ °C | fría, fresca, ambiente, calurosa |
| Consecuente | `modo` (frío → calor) | $[10, 45]$ | frio, standby, calor |
| Consecuente | `vel` (velocidad) | $[0.1, 2.0]$ | baja, media, alta |

**Pasos:** (1) definir variables y universos → (2) asignar funciones de membresía → (3) escribir las reglas `SI…ENTONCES` → (4) ensamblar el `ControlSystem` → (5) simular un instante → (6) iterar en el tiempo (parte 2).

In [ ]:
# Incluir librerías
import numpy as np
import skfuzzy as fuzz
from skfuzzy import control as ctrl

In [ ]:
# Variable de Entradas o Antecedentes
dif = ctrl.Antecedent(np.arange(-10, 10, 0.1), 'dif') # negativa, cercana, positiva
temp = ctrl.Antecedent(np.arange(0, 40, 0.1), 'temp') # fría, fresca, ambiente, calurosa

#Variables de salida o Consecuentes
modo = ctrl.Consequent(np.arange(10, 45, 0.1), 'modo') # frio, standby, calor
vel = ctrl.Consequent(np.arange(0.1, 2.0, 0.1), 'vel') # baja, media, alta

In [ ]:
# Funciones de membresía
dif['negativa'] = fuzz.zmf(dif.universe, -10, 0)
dif['cercana'] = fuzz.gaussmf(dif.universe, 0, 5)
dif['positiva'] = fuzz.smf(dif.universe, 0, 10)
dif.view()

In [ ]:
# Membresías de la temperatura actual
temp['fria'] = fuzz.zmf(temp.universe, 0, 15)
temp['fresca'] = fuzz.trimf(temp.universe, [12.5, 16.25, 20])
temp['ambiente'] = fuzz.trimf(temp.universe, [18, 21.5, 25])
temp['calurosa'] = fuzz.smf(temp.universe, 23, 40)

temp.view()

In [ ]:
# Membresías del modo (automáticas) y ajuste manual del stand-by
modo_names=['frio', 'standby', 'calor']
modo.automf(names=modo_names)
modo['standby'] = fuzz.gaussmf(modo.universe, 13 + (42-13)/2, 2)
modo.view()

In [ ]:
# Membresías de la velocidad (generadas con automf)
vel_names = ['baja', 'media', 'alta']

vel.automf(names=vel_names) 
vel.view()

### Reglas difusas

Traducimos las reglas de negocio del enunciado a reglas `SI … ENTONCES`.

**Ahorro energético (según la diferencia de temperatura):**
* Si la diferencia es negativa → modo **calor** (falta temperatura).
* Si la diferencia es positiva → modo **frío** (sobra temperatura).
* Si la diferencia es cercana a cero → modo **stand-by** y velocidad **baja**.
* Si la diferencia es negativa o positiva (lejos del objetivo) → velocidad **alta**.

**Confort humano (según la temperatura ambiente):**
* Si la temperatura es fría → modo **calor**.
* Si la temperatura es calurosa → modo **frío**.

In [ ]:
# Cada regla relaciona antecedentes con consecuentes. Los resuelve segun su valor de la función de membresía.
regla1 = ctrl.Rule(dif['negativa'], modo['calor'])
regla2 = ctrl.Rule(dif['positiva'], modo['frio'])
regla3 = ctrl.Rule(dif['cercana'], modo['standby'])
regla4 = ctrl.Rule(dif['cercana'], vel['baja'])
regla5 = ctrl.Rule(dif['negativa'] | dif['positiva'], vel['alta'])
regla6 = ctrl.Rule(temp['fria'], modo['calor'])
regla7 = ctrl.Rule(temp['calurosa'], modo['frio'])

In [ ]:
#Creamos un Sistema de Control Difuso, el cual combina todas las reglas creadas.
control_aire = ctrl.ControlSystem([regla1, regla2, regla3, regla4, regla5, regla6, regla7])

In [ ]:
# Cálculo previo de dif temperatura
t_solicitada = 19
t_actual = 25
dif_t = t_actual - t_solicitada

# Simulación
aire_funcionando = ctrl.ControlSystemSimulation(control_aire)

aire_funcionando.input['dif'] = dif_t
aire_funcionando.input['temp'] = t_actual
aire_funcionando.compute()

In [ ]:
# Salida defuzzificada: modo y velocidad
print("Valor de modo", aire_funcionando.output['modo'], "Valor de velocidad", aire_funcionando.output['vel'])

In [ ]:
# Visualización de las membresías activadas por la simulación
modo.view(sim=aire_funcionando)
vel.view(sim=aire_funcionando)

2. Simule un sistema de control difuso que muestre cómo funcionaría el dispositivo durante **15 minutos**, actualizando las condiciones cada **30 segundos** (30 pasos).

**Idea de la simulación:** en cada paso leemos la `velocidad` que entrega el controlador y actualizamos la temperatura ambiente acercándola a la deseada (subiendo si falta calor, bajando si sobra), recalculamos la `diferencia` y volvemos a evaluar el FIS. Así observamos cómo el sistema converge hacia la temperatura objetivo.

In [ ]:
# Parte 2: simulación temporal (15 min, paso de 30 s)
for i in np.arange(0, 15, 0.5):
    if not(25 < aire_funcionando.output['modo'] < 30 or dif_t == 0):
        if dif_t > 0:
            t_actual -= aire_funcionando.output['vel']/2 
        elif dif_t < 0:
            t_actual += aire_funcionando.output['vel']/2
    
    dif_t = t_actual - t_solicitada
    aire_funcionando.input['dif'] = dif_t
    aire_funcionando.input['temp'] = t_actual
    aire_funcionando.compute()
    
    print("Valor de modo", aire_funcionando.output['modo'], "Valor de velocidad", aire_funcionando.output['vel'])

In [ ]:
# Estado final de modo y velocidad tras la simulación
modo.view(sim=aire_funcionando)
vel.view(sim=aire_funcionando)

## 📌 Resumen

| Etapa | Herramienta `scikit-fuzzy` |
|-------|----------------------------|
| Definir variables | `ctrl.Antecedent` / `ctrl.Consequent` sobre `np.arange` |
| Funciones de membresía | `fuzz.trimf`, `trapmf`, `gaussmf`, `pimf`, `smf`, `zmf`, `automf` |
| Reglas SI…ENTONCES | `ctrl.Rule(antecedente, consecuente)` (operadores AND `&`, OR) |
| Ensamblar el sistema | `ctrl.ControlSystem([...])` |
| Simular | `ctrl.ControlSystemSimulation` + `.input` / `.compute()` / `.output` |
| Visualizar | `variable.view()` y `variable.view(sim=...)` |

El sistema **fuzzifica** las entradas numéricas, **infiere** con las reglas, **agrega** los consecuentes y **defuzzifica** (por defecto, el centroide) para entregar `modo` y `velocidad` como números concretos.

## 🔗 Próximos pasos
- Volver al laboratorio principal: [02_SciKit_Fuzzy](02_SciKit_Fuzzy.ipynb).
- Módulo siguiente: [03_ComputacionEvolutiva](../03_ComputacionEvolutiva/03_ComputacionEvolutiva.ipynb).

## 📚 Referencias

- **scikit-fuzzy** — Documentación y API oficial: https://pythonhosted.org/scikit-fuzzy/ · Repositorio: https://github.com/scikit-fuzzy/scikit-fuzzy
- **Tutorial de control difuso (*tipping problem*):** https://pythonhosted.org/scikit-fuzzy/auto_examples/plot_tipping_problem_newapi.html
- **Zadeh, L. A. (1965).** *Fuzzy Sets*. Information and Control, 8(3), 338–353.
- **Mamdani, E. H., & Assilian, S. (1975).** *An experiment in linguistic synthesis with a fuzzy logic controller*. Int. J. Man-Machine Studies, 7(1), 1–13.